In [0]:
%sql
SELECT 
COUNT(DISTINCT t.customer_id) * 1.0 /
COUNT(DISTINCT i.customer_id)
FROM relp.bronze.bronze_interactions i
LEFT JOIN relp.bronze.bronze_transactions t
ON i.customer_id = t.customer_id;

In [0]:
customers = spark.table("relp.silver.silver_customers")
interactions = spark.table("relp.bronze.bronze_interactions")
transactions = spark.table("relp.bronze.bronze_transactions")
listings = spark.table("relp.bronze.bronze_listings")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS relp.gold")
spark.sql("USE relp.gold")

In [0]:
customers.selectExpr("count(distinct customer_id) as total_customers") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_total_customers")

In [0]:
interactions.selectExpr("count(distinct customer_id) as active_customers") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_active_customers")

In [0]:
from pyspark.sql.functions import count

interactions.groupBy("customer_id") \
    .agg(count("*").alias("interaction_count")) \
    .filter("interaction_count > 1") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_retained_customers")

In [0]:
from pyspark.sql.functions import sum

transactions.groupBy("customer_id") \
    .agg(sum("deal_price").alias("total_spent")) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_clv")

In [0]:
transactions.selectExpr("sum(deal_price) as total_revenue") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_total_revenue")

In [0]:
transactions.selectExpr("avg(deal_price) as avg_transaction_value") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_avg_transaction")

In [0]:
from pyspark.sql.functions import date_format, sum

transactions.withColumn("month", date_format("deal_date", "yyyy-MM")) \
    .groupBy("month") \
    .agg(sum("deal_price").alias("total_amount")) \
    .orderBy("month") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_monthly_revenue")

In [0]:
from pyspark.sql.functions import count

transactions.groupBy("agent_id") \
    .agg(count("*").alias("deals_closed")) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_deals_per_agent")

In [0]:
transactions.groupBy("agent_id") \
    .agg(sum("deal_price").alias("revenue")) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_revenue_per_agent")

In [0]:
from pyspark.sql.functions import col

transactions.groupBy("agent_id") \
    .agg(sum("deal_price").alias("revenue")) \
    .orderBy(col("revenue").desc()) \
    .limit(5) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_top_agents")

In [0]:
spark.createDataFrame([(listings.count(),)], ["total_listings"]) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_total_listings")

In [0]:
from pyspark.sql.functions import countDistinct

transactions.select(countDistinct("property_id").alias("sold_properties")) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_sold_properties")

In [0]:
from pyspark.sql.functions import countDistinct

listings.alias("l") \
    .join(transactions.alias("t"), "property_id", "left") \
    .agg(
        (countDistinct("t.property_id") / countDistinct("l.property_id"))
        .alias("conversion_rate")
    ) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_listing_conversion")

In [0]:
spark.createDataFrame([(interactions.count(),)], ["total_interactions"]) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_total_interactions")

In [0]:
interactions.alias("i") \
    .join(transactions.alias("t"), "customer_id", "left") \
    .agg(
        (countDistinct("t.customer_id") / countDistinct("i.customer_id"))
        .alias("conversion_rate")
    ) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("relp.gold.kpi_interaction_conversion")